# ETF Builder BRVM — Notebook de tests

Ce notebook reproduit la logique de l'app `etf_builder.py` sans Streamlit, pour expérimenter les méthodes de pondération avec les contraintes habituelles :
- Plafond de poids par titre
- Long-only, somme des poids = 1
- Filtrage par univers (toutes BRVM ou par secteur)
- Sélection des `n_max` titres les plus liquides
- Backtest avec drift et rebalancement périodique

**Méthodes** : Équipondéré · Capi-pondéré · Free-float pondéré · Inverse de la volatilité · Tracking error minimum · Sharpe maximal.

Reqs : `pandas`, `numpy`, `scipy`, `plotly` (ou `matplotlib`).

## 1. Imports et chemins

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

import plotly.express as px
import plotly.graph_objects as go

from brvm_tickers import ACTIONS, INDICES, safe_filename

DATA_DIR = Path('data')
ACTIONS_DIR = DATA_DIR / 'actions'
INDICES_DIR = DATA_DIR / 'indices'
SOCIETES_DIR = DATA_DIR / 'societes'

NUM_COLS = ['Ouverture', 'Plus_Haut', 'Plus_Bas', 'Cloture', 'Variation_Pct']

print('Univers :', len(ACTIONS), 'actions —', len(INDICES), 'indices')

## 2. Fonctions de chargement des données

In [ ]:
def _read_brvm_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8')
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'Volume_Titres' in df.columns:
        df['Volume_Titres'] = pd.to_numeric(df['Volume_Titres'], errors='coerce')
    return df


def load_action(ticker: str) -> pd.DataFrame | None:
    path = ACTIONS_DIR / f'{safe_filename(ticker)}_historique.csv'
    if not path.exists():
        return None
    df = _read_brvm_csv(path)
    return df[['Date', 'Cloture', 'Volume_Titres']].rename(
        columns={'Cloture': ticker, 'Volume_Titres': f'{ticker}_vol'}
    )


def load_index(symbol: str) -> pd.DataFrame | None:
    path = INDICES_DIR / f'{safe_filename(symbol)}_historique.csv'
    if not path.exists():
        return None
    df = _read_brvm_csv(path)
    return df[['Date', 'Cloture']].rename(columns={'Cloture': symbol})


def _to_float(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    if isinstance(s, (int, float)):
        return float(s)
    s = str(s).replace('\xa0', '').replace(' ', '').replace('%', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return None


def load_societes() -> pd.DataFrame:
    rows = []
    for path in SOCIETES_DIR.glob('*_societe.json'):
        try:
            rows.append(json.loads(path.read_text(encoding='utf-8')))
        except Exception:
            continue
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['Nb_Titres'] = df.get('Nombre_Actions', df.get('Nombre de titres')).apply(_to_float)
    df['Flottant_Pct_num'] = df.get('Flottant_Pct').apply(_to_float)
    df['BNPA_num'] = df.get('BNPA').apply(_to_float)
    df['PER_num'] = df.get('PER').apply(_to_float)
    df['Dividende_num'] = df.get('Dividende').apply(_to_float)
    df = df.rename(columns={'ticker': 'Ticker'})
    return df


def build_prices_panel(tickers: list[str]) -> pd.DataFrame:
    frames = []
    for t in tickers:
        df = load_action(t)
        if df is None:
            continue
        frames.append(df[['Date', t]].set_index('Date'))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, axis=1).sort_index()

## 3. Fonctions de pondération

In [ ]:
def equal_weights(tickers: list[str]) -> pd.Series:
    n = len(tickers)
    return pd.Series(1.0 / n, index=tickers)


def market_cap_weights(tickers, societes, prices) -> pd.Series:
    last_px = prices[tickers].ffill().iloc[-1]
    nb = societes.set_index('Ticker').reindex(tickers)['Nb_Titres']
    cap = (last_px * nb).fillna(0.0)
    if cap.sum() <= 0:
        return equal_weights(tickers)
    return cap / cap.sum()


def free_float_weights(tickers, societes, prices) -> pd.Series:
    last_px = prices[tickers].ffill().iloc[-1]
    soc = societes.set_index('Ticker').reindex(tickers)
    nb = soc['Nb_Titres'].fillna(0.0)
    flottant = (soc['Flottant_Pct_num'].fillna(100.0) / 100.0).clip(0, 1)
    cap_ff = last_px * nb * flottant
    if cap_ff.sum() <= 0:
        return equal_weights(tickers)
    return cap_ff / cap_ff.sum()


def inverse_vol_weights(tickers, returns) -> pd.Series:
    vol = returns[tickers].std()
    inv = (1.0 / vol.replace(0, np.nan)).fillna(0.0)
    if inv.sum() <= 0:
        return equal_weights(tickers)
    return inv / inv.sum()


def min_tracking_error_weights(tickers, asset_returns, bench_returns, max_weight=0.30) -> pd.Series:
    R = asset_returns[tickers].dropna(how='all').fillna(0.0)
    b = bench_returns.reindex(R.index).fillna(0.0).values
    A = R.values
    n = len(tickers)

    def te(w):
        return float(np.std(A @ w - b))

    cons = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_weight)] * n
    x0 = np.full(n, 1.0 / n)
    res = minimize(te, x0, method='SLSQP', bounds=bounds, constraints=cons,
                   options={'maxiter': 300, 'ftol': 1e-9})
    w = res.x if res.success else x0
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.full(n, 1.0 / n)
    return pd.Series(w / w.sum(), index=tickers)


def max_sharpe_weights(tickers, returns, rf_annual=0.03, max_weight=0.30) -> pd.Series:
    R = returns[tickers].dropna(how='all').fillna(0.0)
    mu = R.mean().values * 252
    cov = R.cov().values * 252
    n = len(tickers)

    def neg_sharpe(w):
        ret = float(w @ mu)
        vol = float(np.sqrt(w @ cov @ w))
        if vol <= 0:
            return 1e6
        return -(ret - rf_annual) / vol

    cons = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_weight)] * n
    x0 = np.full(n, 1.0 / n)
    res = minimize(neg_sharpe, x0, method='SLSQP', bounds=bounds, constraints=cons,
                   options={'maxiter': 300, 'ftol': 1e-9})
    w = res.x if res.success else x0
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.full(n, 1.0 / n)
    return pd.Series(w / w.sum(), index=tickers)

## 4. Backtest avec drift + rebalancement et métriques

In [ ]:
def rebalance_freq_map(label):
    return {
        'Aucun (buy & hold)': None,
        'Mensuel': 'ME',
        'Trimestriel': 'QE',
        'Semestriel': '2QE',
        'Annuel': 'YE',
    }[label]


def backtest_with_rebalance(asset_returns, target_weights, freq):
    if freq is None:
        cols = list(target_weights.index)
        w = target_weights.values
        return pd.Series(asset_returns[cols].fillna(0.0).values @ w,
                         index=asset_returns.index, name='ETF')

    rebal_dates = pd.date_range(asset_returns.index.min(), asset_returns.index.max(), freq=freq)
    rebal_dates = asset_returns.index[asset_returns.index.searchsorted(rebal_dates).clip(0, len(asset_returns) - 1)]
    rebal_set = set(pd.DatetimeIndex(sorted(set(rebal_dates))))

    cols = list(target_weights.index)
    R = asset_returns[cols].fillna(0.0).values
    w = target_weights.values.copy()
    pf_ret = np.zeros(R.shape[0])

    for i, dt in enumerate(asset_returns.index):
        r = R[i]
        pf_ret[i] = float(w @ r)
        new_val = w * (1.0 + r)
        s = new_val.sum()
        w = new_val / s if s > 0 else target_weights.values.copy()
        if dt in rebal_set:
            w = target_weights.values.copy()

    return pd.Series(pf_ret, index=asset_returns.index, name='ETF')


def cumulative_curve(rets, base=100.0):
    return base * (1.0 + rets.fillna(0.0)).cumprod()


def annualized_return(rets, periods=252):
    if len(rets) == 0:
        return 0.0
    total = (1.0 + rets.fillna(0.0)).prod()
    years = len(rets) / periods
    return float(total ** (1 / years) - 1) if years > 0 else 0.0


def annualized_vol(rets, periods=252):
    return float(rets.std() * np.sqrt(periods))


def max_drawdown(curve):
    return float((curve / curve.cummax() - 1.0).min())


def tracking_error(etf_rets, bench_rets, periods=252):
    diff = (etf_rets - bench_rets).dropna()
    return float(diff.std() * np.sqrt(periods)) if len(diff) else 0.0


def beta_alpha(etf_rets, bench_rets, periods=252):
    df = pd.concat([etf_rets, bench_rets], axis=1).dropna()
    if len(df) < 2:
        return 0.0, 0.0
    cov = df.cov().iloc[0, 1]
    var = df.iloc[:, 1].var()
    beta = float(cov / var) if var > 0 else 0.0
    alpha = float(df.iloc[:, 0].mean() - beta * df.iloc[:, 1].mean()) * periods
    return beta, alpha


def perf_summary(etf_rets, bench_rets, rf=0.03):
    etf_curve = cumulative_curve(etf_rets)
    bench_curve = cumulative_curve(bench_rets)
    ann_etf = annualized_return(etf_rets)
    vol_etf = annualized_vol(etf_rets)
    beta, alpha = beta_alpha(etf_rets, bench_rets)
    return pd.Series({
        'Perf annualisée ETF': ann_etf,
        'Perf annualisée Bench': annualized_return(bench_rets),
        'Vol annualisée ETF': vol_etf,
        'Vol annualisée Bench': annualized_vol(bench_rets),
        'Tracking Error': tracking_error(etf_rets, bench_rets),
        'Sharpe ETF': (ann_etf - rf) / vol_etf if vol_etf > 0 else 0.0,
        'Beta': beta,
        'Alpha annualisé': alpha,
        'Max Drawdown': max_drawdown(etf_curve),
        'Jours': len(etf_rets),
    })

## 5. Paramètres du test

Modifie les valeurs ci-dessous pour explorer.

In [ ]:
# ===== Paramètres =====
BENCH_SYMBOL = 'BRVM30'              # ex : BRVM30, BRVMC, BRVMPR, BRVM-CB, BRVM-SF, ...
FILTER_SECTOR = None                  # None = toutes ; sinon liste de secteurs (voir cellule plus bas)
N_MAX = 15                            # nb max de titres dans l'ETF
MAX_WEIGHT = 0.25                     # plafond par titre
START = '2024-01-01'
END = '2026-05-05'
REBALANCE = 'Trimestriel'             # Aucun (buy & hold) | Mensuel | Trimestriel | Semestriel | Annuel
RF = 0.03                             # taux sans risque annuel

# Méthodes à comparer
METHODS = [
    'Équipondéré',
    'Capi-pondéré',
    'Free-float pondéré',
    'Inverse de la volatilité',
    'Tracking error min (vs indice)',
    'Sharpe maximal',
]

## 6. Chargement des données

In [ ]:
societes = load_societes()
print('Sociétés :', len(societes))
print('Secteurs disponibles :')
for s in sorted(societes['Secteur'].dropna().unique().tolist()):
    print('  -', s)

In [ ]:
# Univers selon filtre sectoriel
if FILTER_SECTOR:
    universe = societes[societes['Secteur'].isin(FILTER_SECTOR)]['Ticker'].dropna().tolist()
else:
    universe = ACTIONS
print('Univers :', len(universe), 'tickers')

# Benchmark
bench_df = load_index(BENCH_SYMBOL)
assert bench_df is not None and not bench_df.empty, f'Indice {BENCH_SYMBOL} introuvable'
bench_df = bench_df.set_index('Date').loc[START:END]
print(f'{BENCH_SYMBOL} : {len(bench_df)} dates entre {bench_df.index.min().date()} et {bench_df.index.max().date()}')

# Panel des cours
prices = build_prices_panel(universe)
prices = prices.loc[START:END]
common = bench_df.index.intersection(prices.index)
prices = prices.reindex(common).ffill()
bench_df = bench_df.reindex(common).ffill()
print('Panel aligné :', prices.shape)

asset_returns = prices.pct_change().dropna(how='all')
bench_rets = bench_df[BENCH_SYMBOL].pct_change().dropna()

## 7. Sélection des titres (top N par liquidité)

On retient les `N_MAX` actions ayant le plus de jours de cotation sur la période — proxy de liquidité utilisée par l'app.

In [ ]:
ranking = asset_returns.apply(lambda s: s.notna().sum(), axis=0).sort_values(ascending=False)
selected = [t for t in ranking.index if t in universe][:N_MAX]
print(f'{len(selected)} titres retenus :')
print(selected)

# Sous-panel utilisé pour les calculs
asset_returns_sel = asset_returns[selected].dropna(how='all')
asset_returns_sel.tail()

## 8. Calculer une méthode unique

Choisis une méthode et inspecte les poids.

In [ ]:
def compute_weights(method, tickers, asset_returns, bench_rets, prices, societes, max_weight, rf):
    if method == 'Équipondéré':
        w = equal_weights(tickers)
    elif method == 'Capi-pondéré':
        w = market_cap_weights(tickers, societes, prices)
    elif method == 'Free-float pondéré':
        w = free_float_weights(tickers, societes, prices)
    elif method == 'Inverse de la volatilité':
        w = inverse_vol_weights(tickers, asset_returns)
    elif method == 'Tracking error min (vs indice)':
        w = min_tracking_error_weights(tickers, asset_returns, bench_rets, max_weight=max_weight)
    elif method == 'Sharpe maximal':
        w = max_sharpe_weights(tickers, asset_returns, rf_annual=rf, max_weight=max_weight)
    else:
        raise ValueError(method)
    # Application du plafond pour méthodes non-optim
    if method in {'Équipondéré', 'Capi-pondéré', 'Free-float pondéré', 'Inverse de la volatilité'}:
        w = w.clip(upper=max_weight)
        w = w / w.sum()
    return w


method_test = 'Tracking error min (vs indice)'
weights = compute_weights(method_test, selected, asset_returns_sel, bench_rets,
                          prices, societes, MAX_WEIGHT, RF)
weights_df = (pd.DataFrame({'Poids': weights})
              .assign(**{'Poids %': lambda d: (d['Poids'] * 100).round(2)})
              .sort_values('Poids', ascending=False))
weights_df

In [ ]:
# Backtest
etf_rets = backtest_with_rebalance(asset_returns_sel, weights, rebalance_freq_map(REBALANCE))
common_idx = etf_rets.index.intersection(bench_rets.index)
etf_rets = etf_rets.loc[common_idx]
bench_rets_c = bench_rets.loc[common_idx]

perf_summary(etf_rets, bench_rets_c, rf=RF).to_frame(method_test)

In [ ]:
# Courbe de performance
etf_curve = cumulative_curve(etf_rets)
bench_curve = cumulative_curve(bench_rets_c)
curve_df = pd.DataFrame({method_test: etf_curve, BENCH_SYMBOL: bench_curve}).dropna()
fig = px.line(curve_df, title=f'{method_test} vs {BENCH_SYMBOL} — base 100')
fig.show()

## 9. Comparer toutes les méthodes

Backtest des 6 méthodes sur la même période et même univers retenu, avec les contraintes (`MAX_WEIGHT`, rebalancement).

In [ ]:
results_rets = {}
results_weights = {}
results_metrics = {}

for m in METHODS:
    w = compute_weights(m, selected, asset_returns_sel, bench_rets,
                        prices, societes, MAX_WEIGHT, RF)
    rets = backtest_with_rebalance(asset_returns_sel, w, rebalance_freq_map(REBALANCE))
    idx = rets.index.intersection(bench_rets.index)
    rets = rets.loc[idx]
    bench_c = bench_rets.loc[idx]
    results_rets[m] = rets
    results_weights[m] = w
    results_metrics[m] = perf_summary(rets, bench_c, rf=RF)

metrics_df = pd.DataFrame(results_metrics)
for col in metrics_df.columns:
    metrics_df[col] = metrics_df[col].astype(float)

# Format présentation
fmt = metrics_df.copy()
for row in ['Perf annualisée ETF', 'Perf annualisée Bench', 'Vol annualisée ETF',
            'Vol annualisée Bench', 'Tracking Error', 'Alpha annualisé', 'Max Drawdown']:
    fmt.loc[row] = fmt.loc[row].map(lambda x: f'{x*100:.2f}%')
for row in ['Sharpe ETF', 'Beta']:
    fmt.loc[row] = fmt.loc[row].map(lambda x: f'{x:.2f}')
fmt.loc['Jours'] = fmt.loc['Jours'].astype(int)
fmt

In [ ]:
# Courbes comparatives base 100
curves = pd.DataFrame({m: cumulative_curve(r) for m, r in results_rets.items()})
curves[BENCH_SYMBOL] = cumulative_curve(bench_rets.loc[curves.index])
fig = px.line(curves.dropna(), title=f'Comparaison des méthodes — base 100 (vs {BENCH_SYMBOL})')
fig.update_layout(legend_title=None, yaxis_title='Base 100')
fig.show()

In [ ]:
# Heatmap des poids par méthode
weights_table = pd.DataFrame(results_weights).fillna(0)
fig = px.imshow(
    (weights_table * 100).round(2),
    aspect='auto',
    color_continuous_scale='Blues',
    title='Pondérations (%) par méthode',
    labels=dict(x='Méthode', y='Ticker', color='Poids %'),
)
fig.show()

In [ ]:
# Drawdowns comparés
dd_df = pd.DataFrame({
    m: cumulative_curve(r) / cumulative_curve(r).cummax() - 1.0
    for m, r in results_rets.items()
})
fig = px.line(dd_df * 100, title='Drawdown (%) par méthode')
fig.update_layout(yaxis_title='%', legend_title=None)
fig.show()

## 10. Test rapide : sensibilité au plafond `MAX_WEIGHT`

Comparer la tracking error min sur plusieurs niveaux de cap — utile pour calibrer la concentration.

In [ ]:
rows = []
for cap in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    w = min_tracking_error_weights(selected, asset_returns_sel, bench_rets, max_weight=cap)
    rets = backtest_with_rebalance(asset_returns_sel, w, rebalance_freq_map(REBALANCE))
    idx = rets.index.intersection(bench_rets.index)
    rets = rets.loc[idx]; b = bench_rets.loc[idx]
    s = perf_summary(rets, b, rf=RF)
    s['cap'] = cap
    s['n_eff'] = float((w > 1e-4).sum())
    rows.append(s)

sensi = pd.DataFrame(rows).set_index('cap')
sensi[['Perf annualisée ETF', 'Vol annualisée ETF', 'Tracking Error', 'Beta', 'Max Drawdown', 'n_eff']]

## 11. Sensibilité au nombre de titres `N_MAX` (tracking error min)

In [ ]:
rows = []
for n in [5, 8, 10, 15, 20, 25, 30]:
    sel_n = [t for t in ranking.index if t in universe][:n]
    R = asset_returns[sel_n].dropna(how='all')
    w = min_tracking_error_weights(sel_n, R, bench_rets, max_weight=MAX_WEIGHT)
    rets = backtest_with_rebalance(R, w, rebalance_freq_map(REBALANCE))
    idx = rets.index.intersection(bench_rets.index)
    rets = rets.loc[idx]; b = bench_rets.loc[idx]
    s = perf_summary(rets, b, rf=RF)
    s['n_max'] = n
    rows.append(s)

sensi_n = pd.DataFrame(rows).set_index('n_max')
sensi_n[['Perf annualisée ETF', 'Vol annualisée ETF', 'Tracking Error', 'Beta', 'Max Drawdown']]

## 12. Export CSV des pondérations

In [ ]:
out = weights_table.copy()
out.index.name = 'Ticker'
out.to_csv('etf_weights_comparaison.csv', float_format='%.6f')
print('Exporté : etf_weights_comparaison.csv')